In [1]:
# ============================================================
# FINANCE TRACKER PROJECT
# Complete Jupyter Notebook Code
# ============================================================

import os
import json
import pandas as pd
import numpy as np
from datetime import datetime
from groq import Groq
from IPython.display import display

print("=" * 55)
print("             FINANCE TRACKER PROJECT")
print("=" * 55)

# ============================================================
# 1. TRANSACTION DATA
# ============================================================
# Maximum transactions allowed
MAX_TRANSACTIONS = 4
transactions = []

# Load existing transactions from Excel if available
EXCEL_FILE = "finance_data_final.xlsx"

if os.path.exists(EXCEL_FILE):
    try:
        df_existing = pd.read_excel(EXCEL_FILE)
        required_columns = ["id", "type", "category", "amount", "date"]

        if all(col in df_existing.columns for col in required_columns):
            transactions = df_existing[required_columns].to_dict("records")
            print("Existing transactions loaded from Excel.")
        else:
            print("Excel file has missing columns. Starting with empty data.")

    except Exception as e:
        print("Could not load existing Excel data:", e)

# ============================================================
# 2. VALIDATE TRANSACTIONS
# ============================================================

def add_transaction():

    if len(transactions) >= MAX_TRANSACTIONS:
        print(
            f"\n❌ Maximum limit of {MAX_TRANSACTIONS} transactions reached."
        )
        print(
            "Please delete an existing transaction before adding a new one."
        )
        return

    try:

        transaction_id = input(
            "Enter transaction ID: "
        ).strip()

        if any(
            t["id"] == transaction_id
            for t in transactions
        ):
            print(
                "Duplicate transaction ID! Use another ID."
            )
            return

        transaction_type = input(
            "Enter type (Income/Expense): "
        ).strip().title()

        category = input(
            "Enter category: "
        ).strip()

        amount = float(
            input("Enter amount: ")
        )

        date = input(
            "Enter date (DD-MM-YYYY): "
        ).strip()

        transaction = {
            "id": transaction_id,
            "type": transaction_type,
            "category": category,
            "amount": amount,
            "date": date
        }

        if validate_transaction(transaction):

            transactions.append(transaction)

            export_to_excel()

            print(
                "\n✅ Transaction added successfully!"
            )

            print(
                f"Transactions Used: "
                f"{len(transactions)}/{MAX_TRANSACTIONS}"
            )

    except ValueError:
        print(
            "Invalid amount. Please enter a number."
        )

    except Exception as e:
        print(
            "Error adding transaction:",
            e
        )


# ============================================================
# 3. EXCEL EXPORT
# ============================================================

def export_to_excel(filename=EXCEL_FILE):
    try:
        columns = ["id", "type", "category", "amount", "date"]
        df = pd.DataFrame(transactions, columns=columns)
        df.to_excel(filename, index=False)
        print(f"Transactions exported to {filename} successfully.")

    except PermissionError:
        print("Permission denied. Close the Excel file and try again.")

    except Exception as e:
        print("Excel export error:", e)


# ============================================================
# 4. ADD TRANSACTION
# ============================================================

def add_transaction():
    try:
        transaction_id = input("Enter transaction ID: ").strip()

        if any(t["id"] == transaction_id for t in transactions):
            print("Duplicate transaction ID! Use another ID.")
            return

        transaction_type = input("Enter type (Income/Expense): ").strip().title()
        category = input("Enter category: ").strip()
        amount = float(input("Enter amount: "))
        date = input("Enter date (DD-MM-YYYY): ").strip()

        transaction = {
            "id": transaction_id,
            "type": transaction_type,
            "category": category,
            "amount": amount,
            "date": date
        }

        if validate_transaction(transaction):
            transactions.append(transaction)
            export_to_excel()
            print("Transaction added successfully!")

    except ValueError:
        print("Invalid amount. Please enter a number.")

    except Exception as e:
        print("Error adding transaction:", e)


# ============================================================
# 5. VIEW TRANSACTIONS
# ============================================================

def view_transactions():
    if not transactions:
        print("No transactions found.")
        return

    display(pd.DataFrame(transactions))


# ============================================================
# 6. FINANCIAL SUMMARY
# ============================================================

def view_summary():
    income = sum(
        t["amount"] for t in transactions
        if t["type"] == "Income"
    )

    expenses = sum(
        t["amount"] for t in transactions
        if t["type"] == "Expense"
    )

    print("\n========== FINANCIAL SUMMARY ==========")
    print(f"Total Income: ₹{income:.2f}")
    print(f"Total Expenses: ₹{expenses:.2f}")
    print(f"Remaining Balance: ₹{income - expenses:.2f}")


# ============================================================
# 7. SPENDING BY CATEGORY
# ============================================================

def spending_by_category():
    expenses = [
        t for t in transactions
        if t["type"] == "Expense"
    ]

    if not expenses:
        print("No expense transactions found.")
        return

    df = pd.DataFrame(expenses)
    category_totals = df.groupby("category")["amount"].sum()

    print("\n========== SPENDING BY CATEGORY ==========")
    display(category_totals)

    highest_category = category_totals.idxmax()
    highest_amount = category_totals.max()

    print("\n========== HIGHEST EXPENSE ==========")
    print("Category:", highest_category)
    print(f"Amount: ₹{highest_amount:.2f}")


# ============================================================
# 8. DELETE TRANSACTION
# ============================================================

def delete_transaction():
    transaction_id = input("Enter transaction ID to delete: ").strip()

    for transaction in transactions:
        if transaction["id"] == transaction_id:
            transactions.remove(transaction)
            export_to_excel()
            print("Transaction deleted successfully.")
            return

    print("Transaction ID not found.")


# ============================================================
# 9. DELETE ALL TRANSACTIONS
# ============================================================

def delete_all_transactions():
    confirmation = input(
        "Type DELETE ALL to permanently remove all transactions: "
    )

    if confirmation == "DELETE ALL":
        transactions.clear()
        export_to_excel()
        print("All transactions deleted.")
    else:
        print("Deletion cancelled.")


# ============================================================
# 10. NUMPY ANALYSIS
# ============================================================

def numpy_analysis():
    if not transactions:
        print("No transactions available for analysis.")
        return

    amounts = np.array([t["amount"] for t in transactions])

    print("\n========== NUMPY ANALYSIS ==========")
    print("Transaction amounts:", amounts)
    print("Total transaction amount:", np.sum(amounts))
    print("Average transaction amount:", np.mean(amounts))
    print("Unique amounts:", np.unique(amounts))

    expense_amounts = np.array([
        t["amount"] for t in transactions
        if t["type"] == "Expense"
    ])

    high_expenses = np.where(expense_amounts > 2000)

    print(
        "Expenses greater than ₹2000:",
        expense_amounts[high_expenses]
    )


# ============================================================
# 11. LAMBDA HIGH-EXPENSE DETECTION
# ============================================================

def identify_high_expenses():
    high_expense_checker = lambda amount: (
        "High Expense" if amount > 2000 else "Normal Expense"
    )

    print("\n========== EXPENSE CHECK ==========")

    for t in transactions:
        if t["type"] == "Expense":
            print(
                t["category"],
                "- ₹" + str(t["amount"]),
                "-",
                high_expense_checker(t["amount"])
            )


# ============================================================
# 12. FINANCIAL PROFILE
# ============================================================

def classify_financial_profile():
    income = sum(
        t["amount"] for t in transactions
        if t["type"] == "Income"
    )

    expenses = sum(
        t["amount"] for t in transactions
        if t["type"] == "Expense"
    )

    if income <= 0:
        profile = "No Income"
    else:
        ratio = expenses / income

        if ratio < 0.5:
            profile = "Saver"
        elif ratio <= 0.9:
            profile = "Balanced"
        else:
            profile = "Spender"

    print("\nFinancial Profile:", profile)
    return profile


# ============================================================
# 13. TRANSACTION MANAGER CLASS
# ============================================================

class TransactionManager:

    def __init__(self, transactions):
        self.transactions = transactions

    def view_transactions(self):
        if not self.transactions:
            print("No transactions found.")
        else:
            display(pd.DataFrame(self.transactions))

    def calculate_summary(self):
        income = sum(
            t["amount"] for t in self.transactions
            if t["type"] == "Income"
        )

        expenses = sum(
            t["amount"] for t in self.transactions
            if t["type"] == "Expense"
        )

        print("Total Income: ₹", income)
        print("Total Expenses: ₹", expenses)
        print("Remaining Balance: ₹", income - expenses)

    def analyze_spending(self):
        expenses = [
            t for t in self.transactions
            if t["type"] == "Expense"
        ]

        if not expenses:
            print("No expenses available.")
            return

        df = pd.DataFrame(expenses)
        display(df.groupby("category")["amount"].sum())

    def export_to_excel(self):
        export_to_excel()


# ============================================================
# 14. IMPORT FROM EXCEL
# ============================================================

def import_from_excel(filename=EXCEL_FILE):
    global transactions

    try:
        df = pd.read_excel(filename)
        required_columns = ["id", "type", "category", "amount", "date"]

        if not all(col in df.columns for col in required_columns):
            print("Excel file is missing required columns.")
            return

        transactions = df[required_columns].to_dict("records")
        print("Excel data imported successfully.")
        display(df)

    except FileNotFoundError:
        print("Excel file not found.")

    except PermissionError:
        print("Permission denied. Check the Excel file.")

    except Exception as e:
        print("Excel import error:", e)


# ============================================================
# 15. CUSTOM MODULE SUPPORT
# ============================================================

try:
    import finance_utils

    print("Custom finance_utils module imported.")

except ImportError:
    print("Warning: finance_utils.py was not found.")
    print("The project can still use its built-in calculations.")


# ============================================================
# 16. GROQ AI FINANCIAL INSIGHTS
# ============================================================
def get_ai_financial_insights():

    if not transactions:
        print("No transactions available for AI analysis.")
        return

    api_key = os.getenv("GROQ_API_KEY")

    if not api_key:
        print("Please set your GROQ_API_KEY first.")
        return

    income = sum(
        t["amount"]
        for t in transactions
        if t["type"] == "Income"
    )

    expenses = sum(
        t["amount"]
        for t in transactions
        if t["type"] == "Expense"
    )

    balance = income - expenses

    spending = {}

    for t in transactions:
        if t["type"] == "Expense":
            category = t["category"]
            spending[category] = (
                spending.get(category, 0)
                + t["amount"]
            )

    try:

        client = Groq(api_key=api_key)

        print("\n" + "=" * 50)
        print("     AI FINANCIAL ASSISTANT")
        print("=" * 50)
        print("Ask any question.")
        print("Type 'exit' to return to menu.")

        while True:

            user_question = input("\nYou: ").strip()

            if user_question.lower() == "exit":
                print("Returning to menu...")
                break

            prompt = f"""
You are a Personal Finance Assistant.

User Financial Data:
Income: ₹{income}
Expenses: ₹{expenses}
Balance: ₹{balance}
Spending Categories: {spending}

User Question:
{user_question}

Rules:
Rules:
1. Answer in 4-6 lines.
2. Use simple English.
3. Be direct and complete.
4. Do not use tables.
5. Give a complete answer.
"""

            response = client.chat.completions.create(
                model="openai/gpt-oss-120b",
                messages=[
                    {
                        "role": "user",
                        "content": prompt
                    }
                ],
                temperature=0.3,
                max_tokens=200
            )

            print("\nAI Advisor:")
            print(
                response.choices[0].message.content
            )

    except Exception as e:
        print("AI Error:", e)
# ============================================================
# 17. MENU-DRIVEN APPLICATION
# ============================================================

def show_menu():
    print("\n========== FINANCE TRACKER MENU ==========")
    print("1. Add Transaction")
    print("2. View Transactions")
    print("3. View Financial Summary")
    print("4. Spending by Category")
    print("5. Delete Transaction")
    print("6. Delete All Transactions")
    print("7. Import from Excel")
    print("8. Export to Excel")
    print("9. NumPy Analysis")
    print("10. Identify High Expenses")
    print("11. Financial Profile")
    print("12. AI Financial Insights")
    print("13. Exit")


def run_finance_tracker():
    while True:
        show_menu()

        choice = input("Enter your choice (1-13): ").strip()

        if choice == "1":
            add_transaction()

        elif choice == "2":
            view_transactions()

        elif choice == "3":
            view_summary()

        elif choice == "4":
            spending_by_category()

        elif choice == "5":
            delete_transaction()

        elif choice == "6":
            delete_all_transactions()

        elif choice == "7":
            import_from_excel()

        elif choice == "8":
            export_to_excel()

        elif choice == "9":
            numpy_analysis()

        elif choice == "10":
            identify_high_expenses()

        elif choice == "11":
            classify_financial_profile()

        elif choice == "12":
            get_ai_financial_insights()

        elif choice == "13":
            print("Finance Tracker closed. Goodbye!")
            break

        else:
            print("Invalid choice. Please enter a number from 1 to 13.")


# ============================================================
# 18. FINAL PROJECT SUMMARY
# ============================================================

print("\nFinance Tracker code loaded successfully!")
print("Number of transactions loaded:", len(transactions))

if transactions:
    view_summary()

print("\nTo start the interactive application, run:")
print("run_finance_tracker()")

             FINANCE TRACKER PROJECT
Existing transactions loaded from Excel.
Custom finance_utils module imported.

Finance Tracker code loaded successfully!
Number of transactions loaded: 6

========== FINANCIAL SUMMARY ==========
Total Income: ₹23000.00
Total Expenses: ₹7000.00
Remaining Balance: ₹16000.00

To start the interactive application, run:
run_finance_tracker()


In [2]:
import os
import getpass

os.environ["GROQ_API_KEY"] = getpass.getpass(
    "Enter your Groq API key:"
)

print("Groq API key configured for this session!")

Enter your Groq API key: ········


Groq API key configured for this session!


In [3]:
get_ai_financial_insights()



     AI FINANCIAL ASSISTANT
Ask any question.
Type 'exit' to return to menu.



You:  best tips to save money



AI Advisor:
1. Track every expense and compare it with your budget; notice where you overspend and cut it back.  
2. Set a fixed amount (e.g., ₹5,000) each month for savings before you pay any other bills.  
3. Cook at home more often and limit eating out; this can reduce your food cost by 30‑40 %.  
4. Use public transport or car‑pool for travel instead of frequent rides or taxis.  
5. Wait 24 hours before any non‑essential purchase; many impulse buys disappear after the pause.



You:  what is python



AI Advisor:
Python is a high‑level programming language that is easy to read and write.  
It was created by Guido van Rossum and first released in 1991.  
Python supports many programming styles, including procedural, object‑oriented, and functional.  
It has a large standard library and many third‑party packages for tasks like web development, data analysis, and automation.  
Because of its simplicity and versatility, Python is popular among beginners and professionals alike.



You:  exit


Returning to menu...


In [4]:
run_finance_tracker()



========== FINANCE TRACKER MENU ==========
1. Add Transaction
2. View Transactions
3. View Financial Summary
4. Spending by Category
5. Delete Transaction
6. Delete All Transactions
7. Import from Excel
8. Export to Excel
9. NumPy Analysis
10. Identify High Expenses
11. Financial Profile
12. AI Financial Insights
13. Exit


Enter your choice (1-13):  1
Enter transaction ID:  T004353
Enter type (Income/Expense):  income
Enter category:  shopping
Enter amount:  12000
Enter date (DD-MM-YYYY):  12-03-2026


Error adding transaction: name 'validate_transaction' is not defined

========== FINANCE TRACKER MENU ==========
1. Add Transaction
2. View Transactions
3. View Financial Summary
4. Spending by Category
5. Delete Transaction
6. Delete All Transactions
7. Import from Excel
8. Export to Excel
9. NumPy Analysis
10. Identify High Expenses
11. Financial Profile
12. AI Financial Insights
13. Exit


Enter your choice (1-13):  13


Finance Tracker closed. Goodbye!
